# Giáo trình Dữ liệu lớn – Chương 2

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume và thay `data/` bằng `/Volumes/<catalog>/<schema>/<volume>/`.

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch{ch:02d}/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
!pip install -q pyspark==3.5.7 pyarrow==16.1.0 pandas==2.2.2
import os
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch02").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 2.1. Cài đặt và khởi tạo PySpark trên Google Colab.


> Trên Colab, ô này cài PySpark; trên Databricks bỏ dòng `!pip` (Spark đã có sẵn).


In [ ]:
# Buoc 1: kiem tra Java co san tren may ao Colab
!java -version

# Buoc 2: cai dat PySpark tu PyPI (ghim phien ban theo Bang 2.3)
!pip install -q pyspark==3.5.7

# Buoc 3: khoi tao SparkSession o che do cuc bo
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("MoiTruongColab")
         .getOrCreate())

# Buoc 4: kiem chung phien ban va chay thu mot phep dem
print("Phien ban Spark:", spark.version)
so_dong = spark.range(1, 10**7).count()
print("Ket qua dem:", so_dong)

## Đoạn mã 2.2. Đọc tệp CSV đã tải lên volume trong notebook Databricks.


> Đoạn mã dành cho Databricks (đường dẫn volume). Trong notebook này đường dẫn đã đổi thành `data/diem_thi.csv` để chạy được trên Colab.


In [ ]:
# Tren Databricks, doi tuong spark da duoc khoi tao san
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("data/diem_thi.csv"))
df.show(5)

## Đoạn mã 2.3. Ứng dụng PySpark hoàn chỉnh: đọc CSV, thống kê và dừng phiên.


In [ ]:
# Tep: ung_dung_thong_ke.py
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Khoi tao SparkSession bang builder pattern
spark = (SparkSession.builder
         .master("local[*]")
         .appName("ThongKeDiemThi")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

# 2. Doc du lieu CSV co dong tieu de, tu suy dien kieu du lieu
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("data/diem_thi.csv"))

# 3. Khao sat cau truc va thong ke mo ta
df.printSchema()
df.describe("diem_toan", "diem_van").show()

# 4. Dem so thi sinh va tinh diem trung binh theo tinh
ket_qua = (df.groupBy("ma_tinh")
           .agg(F.count("*").alias("so_thi_sinh"),
                F.round(F.avg("diem_toan"), 2)
                 .alias("diem_tb_toan"))
           .orderBy(F.desc("diem_tb_toan")))
ket_qua.show(10)

# 5. Dung phien lam viec, giai phong tai nguyen
# spark.stop()  # giu phien Spark de chay tiep cac o sau

## Đoạn mã 2.4. Nộp ứng dụng lên cụm YARN bằng spark-submit.


> **Khung / minh họa – không chạy trực tiếp:** lệnh shell spark-submit (chạy trên terminal, cần cụm YARN).


In [ ]:
# Lenh chay tren terminal cua may client
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --num-executors 4 \
  --executor-memory 4g \
  --executor-cores 2 \
  ung_dung_thong_ke.py

## Đoạn mã 2.5. Phân phối mã nguồn dùng chung cho các executor.


> Tệp `tien_ich.zip` mẫu nằm trong `code/ch02/`; ô chuẩn bị bên dưới sao chép tệp này vào thư mục làm việc.


In [ ]:
import shutil, os
shutil.copy("code/ch02/tien_ich.zip", "tien_ich.zip")
print("da sao chep tien_ich.zip")

In [ ]:
# Cach 1: khai bao khi nop ung dung
#   spark-submit --py-files tien_ich.zip ung_dung.py

# Cach 2: them tep ngay trong luc chay
spark.sparkContext.addPyFile("tien_ich.zip")
from tien_ich import chuan_hoa_ten  # ham nay dung duoc trong UDF

## Đoạn mã 2.6. Đóng gói môi trường conda và nộp lên cụm YARN.


> **Khung / minh họa – không chạy trực tiếp:** lệnh shell conda-pack (chỉ minh họa).


In [ ]:
# Tren may phat trien: tao va dong goi moi truong conda
#   conda create -y -n mtspark python=3.11 numpy pandas
#   conda pack -n mtspark -o mtspark.tar.gz

# Khi nop ung dung: gui archive kem theo va tro bien moi truong
#   export PYSPARK_DRIVER_PYTHON=python
#   export PYSPARK_PYTHON=./mt/bin/python
#   spark-submit --master yarn \
#     --archives mtspark.tar.gz#mt \
#     ung_dung.py

## Đoạn mã 2.7. Cài đặt theo phiên bản đã ghim và kiểm tra môi trường thực hành.


In [ ]:
# Cai dat dung phien ban da ghim (Colab: dau !, Databricks: %pip)
!pip install -q pyspark==3.5.7 pyarrow==16.1.0 pandas==2.2.2

import sys, platform, subprocess
import pyspark, pandas, pyarrow
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()
jv = subprocess.run(["java", "-version"], text=True,
                    capture_output=True)
java_ver = jv.stderr.splitlines()[0]
print("Python :", sys.version.split()[0], "-", platform.system())
print("Java   :", java_ver)
print("Spark  :", spark.version, "| pyspark", pyspark.__version__)
print("pandas :", pandas.__version__)
print("pyarrow:", pyarrow.__version__)
# Ket qua mong doi: Python 3.11.x, OpenJDK 11 hoac 17, Spark 3.5.7